Copyright 2025 DeepMind Technologies Limited.

In [ ]:
# @title SPDX-License-Identifier: Apache-2.0
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# RAG Tutorial with OneTwo

This tutorial demonstrates how to build a **Retrieval-Augmented Generation (RAG)** system using the OneTwo framework.

Large Language Models (LLMs) are trained on vast amounts of public data, but they often lack access to **private corpora**, domain-specific documents, or real-time information. Furthermore, complex questions often require synthesizing information that is **spread across multiple documents** in a large corpus.

**RAG** solves this by separating the knowledge retrieval from the generation:
1.  **Retrieve**: When asked a question, the system searches a corpus to find the most relevant documents.
2.  **Augment**: The retrieved documents are provided to the LLM as context.
3.  **Generate**: The LLM uses this context to generate an accurate, grounded answer.

In this tutorial, we will use OneTwo's standard interfaces to build a RAG system from scratch. We will cover:
1.  **Basic Retrieval**: Implementing custom retrievers.
2.  **Indexing and Corpus Rewriting**: Preparing documents for semantic search.
3.  **QA Strategies**: Combining retrieval and generation.
4.  **ReAct Agents**: Giving agents the ability to search dynamically.

This tutorial is similar in spirit to the main [tutorial.ipynb](tutorial.ipynb) but focuses specifically on the retrieval features of OneTwo.

# Setup


We will use the `GoogleGenAIAPI` backend for this tutorial. For more details on connecting to other backends (like `OpenAI`) or advanced configuration, please refer to the main `tutorial.ipynb`.

In [ ]:
model_selection = 'Gemini API: gemini-2.5-flash'

OneTwo can connect to publicly-hosted Gemini models via the Gemini API. If you have not used the Gemini API before, you will need to first create an account and API key following the instructions on https://ai.google.dev/. Then either copy-paste your API key into the text box, or store it in the 'GOOGLE_API_KEY' environment variable.

In [ ]:
# You can specify your API key either here or as an environment variable.
google_api_key = ''  # @param {type: 'string'}

if google_api_key:
  print('Using API key specified in the colab.')

if not google_api_key and running_in_colab:
  try:
    google_api_key = userdata.get('GOOGLE_API_KEY')
    print('Loaded GOOGLE_API_KEY from user secrets.')
  except:
    pass

if not google_api_key and 'GOOGLE_API_KEY' in os.environ:
  google_api_key = os.environ['GOOGLE_API_KEY']
  print('Loaded GOOGLE_API_KEY from environment.')

if not google_api_key:
  raise ValueError(
      'The GOOGLE_API_KEY must be specified either here or in a user secret or '
      'in the environment.'
  )

## Imports

In [ ]:
import dataclasses
import pprint

from google.genai import types as genai_types
from onetwo import ot
from onetwo.agents import react
from onetwo.backends import google_genai_api
from onetwo.builtins import llm
from onetwo.core import caching
from onetwo.evaluation import datasets
from onetwo.stdlib.qa import retrieval_qa
from onetwo.stdlib.retrieval import chunking
from onetwo.stdlib.retrieval import corpus_rewriting
from onetwo.stdlib.retrieval import document_formatting
from onetwo.stdlib.retrieval import indexing
from onetwo.stdlib.retrieval import retrieval
from onetwo.stdlib.retrieval import retrieval_data_structures
from onetwo.stdlib.tool_use import llm_tool_use
from onetwo.stdlib.tool_use import python_tool_use
from onetwo.utils import colab_utils

## Caching Configuration

In [ ]:
# Here we define a location in which to store a cache of requests/replies for
# each backend of interest, which we can use for speeding up running of the
# colab if we re-run it (or make iterative modifications to it) in the future.
# We will use a separate cache file for each backend.
OWN_CACHE_DIRECTORY = '/tmp/onetwo_colab_backend_caches/tutorial'

# If you would like to share cache files with others in your working group, you
# can optionally specify another shared cache directory here. If you specify
# this, then we will read from shared cache directory and give precedence to its
# contents, while merging in any additional content from OWN_CACHE_DIRECTORY.
# When saving the cache, however, we will by default write only to
# OWN_CACHE_DIRECTORY to reduce the chance of people clobbering each other's
# changes.
SHARED_CACHE_DIRECTORY = None

# If you want to automatically merge in content from any of your teammates'
# cache directories or from a cache that was output by a batch eval run,
# you can list the additional directories here.
ADDITIONAL_CACHE_DIRECTORIES = []

In [ ]:
# We encapsulate the logic for loading/saving the caches for a set of backends
# in the `CachedBackends` class. Each time we construct a backend, we will add
# it to the `cached_backends` mapping so that we can load and save them all at
# once.
cached_backends = colab_utils.CachedBackends(
    own_cache_directory=OWN_CACHE_DIRECTORY,
    shared_cache_directory=SHARED_CACHE_DIRECTORY,
    additional_cache_directories=ADDITIONAL_CACHE_DIRECTORIES,
)

## Gemini API

In [ ]:
# E.g., 'Gemini API: gemini-1.0-pro' ==> 'gemini-1.0-pro'.
model_name = model_selection.split(':')[-1].strip()

generate_config = {}
# We are disabling safety settings for the purposes of this colab to
# avoid the chance of getting empty responses in the case of
# over-triggering of the safety controls. You can set this to an
# appropriate setting for your use case.
safety_settings = google_genai_api.SAFETY_DISABLED
generate_config['safety_settings'] = safety_settings

# For thinking models, you can control the amount of thinking budget
# using the `ThinkingConfig` below. For the purposes of this colab,
# we are disabling thinking by default, as it tends to not play well
# with the use of the `max_tokens` parameter (i.e.,
# `max_output_tokens` in the GoogleGenAI API), due to it getting
# applied to the sum of the thinking tokens and actual output
# tokens, which can often lead to empty outputs.
thinking_budget = 128
generate_config['thinking_config'] = genai_types.ThinkingConfig(
    include_thoughts=False, thinking_budget=thinking_budget
)


backend = google_genai_api.GoogleGenAIAPI(
    vertexai=False,
    api_key=google_api_key,
    temperature=0.0,
    generate_model_name=f'models/{model_name}',
    chat_model_name=f'models/{model_name}',
    threadpool_size=100,
    max_retries=3,
    cache=caching.SimpleFunctionCache(
        cache_filename=cached_backends.get_cache_path(model_name),
    ),
    generate_text_kwargs=generate_config,
    generate_object_kwargs=generate_config,
    generate_content_kwargs=generate_config,
    chat_kwargs=generate_config,
)

cached_backends[model_name] = backend
backend.register()

print(f'Gemini API backend registered: {model_name}')

# Retriever

In OneTwo, a `Retriever` is a generic interface for any strategy that takes a
query and returns a list of relevant results (typically documents or text
passages). A retriever can be anything from a simple keyword search, a vector
database, or even a web-based search engine!

Below is an example of **Google Search** as a `Retriever`. As
seen in [tutorial.ipynb](tutorial.ipynb), one way to perform Google Search is
leveraging the tool-use capabilities of the
[Google Generative AI SDK](https://github.com/googleapis/python-genai).

In [ ]:
genai_search_tool = genai_types.Tool(google_search=genai_types.GoogleSearch())
chat_config_kwargs = {
    'system_instruction': (
        'You are an expert in Google Search. Use the Search tool for every'
        ' query the user provides.'
    ),
    'tools': [genai_search_tool],
    'max_tokens': 128,
}


@dataclasses.dataclass
class SearchRetriever(retrieval.Retriever[str, str]):

  @ot.make_executable(copy_self=False)
  async def retrieve(
      self,
      query: str,
      *,
      max_results: int | None = None,
      **kwargs,
  ) -> list[str]:
    """Search engine. Returns a relevant snippet or answer to query."""
    result = await llm.instruct(query, **chat_config_kwargs)
    return [result]


search_as_retriever = SearchRetriever()

In [ ]:
query = "What is the capital of France?"
result = ot.run(search_as_retriever(query))
print(f"Result type: {type(result)}")
print(f"Result content: {result}")

# Indexing and Corpus Rewriting

Now that we understand the basics of a `Retriever`, let's look at how we typically build one for a **large collection of documents** (a corpus).

Imagine you have a set of documents containing proprietary knowledge or detailed facts that an LLM wouldn't know by default. To make this information accessible to our RAG system, we need to:
1.  **Organize** the documents in a way that allows fast searching (Indexing).
2.  **Prepare** the documents so that the search is effective (Corpus Rewriting).

## Indexing

In OneTwo, an `Index` is a special kind of `Retriever` that manages a specific corpus of documents. We will use `EmbeddingBasedDocumentIndex`, which enables **semantic search**. Instead of just looking for exact keyword matches, it uses embeddings to find documents that are conceptually similar to the query.

To see this in action, let's create a toy corpus representing "private" knowledge about animals in a zoo that we want our LLM to be able to access.

In [ ]:
# Create a toy corpus of documents.
toy_docs = [
    retrieval_data_structures.Document(
        doc_id="lion",
        content=(
            "Lions are large cats that live in prides. They are apex predators."
        ),
        title="Lion Facts",
    ),
    retrieval_data_structures.Document(
        doc_id="elephant",
        content=(
            "Elephants are the largest land animals. They have long trunks and"
            " tusks."
        ),
        title="Elephant Facts",
    ),
    retrieval_data_structures.Document(
        doc_id="monkey",
        content=(
            "Monkeys are primates that often live in trees. They are known for"
            " their intelligence."
        ),
        title="Monkey Facts",
    ),
]

# Create an empty embedding-based index for Documents.
toy_index = indexing.EmbeddingBasedDocumentIndex()

# Populate the index with our toy docs.
# This will calculate embeddings for all documents.
ot.run(toy_index.create_index(corpus_name="animal_facts", docs=toy_docs))

print(f"Index created with {toy_index.num_docs} documents.")

Let's test the retrieval using our created toy index.

In [ ]:
# Let's test retrieval.
query = "What is the largest land animal?"
results = ot.run(toy_index.retrieve(query, max_results=1))

print(f"Query: {query}")
for doc in results:
  print(f"Retrieved Doc ID: {doc.doc_id}")
  print(f"Content: {doc.content}")

## Corpus Rewriting

Often, raw documents are not in the best format for retrieval. They might be too long, or lack structure. OneTwo provides `CorpusRewriter` to transform document collections before indexing.

We will demonstrate:
1.  **Chunking**: Breaking documents into smaller pieces.
2.  **Formatting**: Adding structure (e.g., titles) to the text.
3.  **Sequential Rewriting**: Combining these operations.

*Note: OneTwo provides built-in chunkers like `ChunkByMaxTokens` which use the LLM backend to split text by tokens. However, this requires the backend to support tokenization. For this tutorial, we will create a custom word-based chunker to avoid this dependency and show how easy it is to extend OneTwo.*

In [ ]:
# Create a custom chunker that splits by words.
# OneTwo's TextChunker handles all the metadata (doc_id, chunk_number) automatically!
@dataclasses.dataclass
class SimpleWordChunker(chunking.TextChunker):
  """Custom chunker that splits text by words."""

  max_words_per_chunk: int = 5
  overlap_words: int = 1

  @ot.make_executable(copy_self=False)
  async def _chunk_text(self, text: str) -> list[str]:
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
      chunk_words = words[i : i + self.max_words_per_chunk]
      chunks.append(" ".join(chunk_words))
      i += self.max_words_per_chunk - self.overlap_words
      if i >= len(words) or len(chunk_words) < self.max_words_per_chunk:
        break
    return chunks


# Use our custom chunker.
simple_chunker = SimpleWordChunker(max_words_per_chunk=5, overlap_words=1)
chunking_rewriter = corpus_rewriting.ChunkingCorpusRewriter(
    chunker=simple_chunker
)

# Rewrite the corpus.
chunked_docs = ot.run(chunking_rewriter(toy_docs))

print(f"Original docs: {len(toy_docs)}")
print(f"Chunked docs: {len(list(chunked_docs))}")
for doc in chunked_docs:
  print(f"Chunk ID: {doc.doc_id}, Content: {doc.content}")

In [ ]:
# Use a formatter to add structure.
simple_formatter = document_formatting.TextDocumentFormatter(
    format_str="Title: {title}\nFacts: {text}",
)
formatting_rewriter = corpus_rewriting.FormattingCorpusRewriter(
    formatter=simple_formatter
)

# Rewrite the corpus.
formatted_docs = ot.run(formatting_rewriter(toy_docs))

print(f"Original docs: {len(toy_docs)}")
print(f"Formatted docs: {len(list(formatted_docs))}")
for doc in formatted_docs:
  print(f"Doc ID: {doc.doc_id}")
  print(f"Content:\n{doc.content}")
  print("-" * 20)

In [ ]:
# Combine them using SequentialCorpusRewriter.
sequential_rewriter = corpus_rewriting.SequentialCorpusRewriter(
    rewriters=[chunking_rewriter, formatting_rewriter]
)

# Rewrite the corpus.
processed_docs = ot.run(sequential_rewriter(toy_docs))

print(f"Original docs: {len(toy_docs)}")
print(f"Processed docs: {len(list(processed_docs))}")
for doc in processed_docs:
  print(f"Doc ID: {doc.doc_id}")
  print(f"Content:\n{doc.content}")
  print("-" * 20)

In [ ]:
advanced_index = indexing.EmbeddingBasedDocumentIndex()

# Populate the index with our processed docs.
ot.run(
    advanced_index.create_index(
        corpus_name="advanced_animal_facts", docs=processed_docs
    )
)
print(f"Advanced index created with {advanced_index.num_docs} documents.")

# Test retrieval on the advanced index.
query = "Tell me about lions."
results = ot.run(advanced_index.retrieve(query, max_results=4))

print(f"\nQuery: {query}")
for doc in results:
  print(f"Retrieved Doc ID: {doc.doc_id}")
  print(f"Content:\n{doc.content}")
  print("-" * 20)

# QA Strategy and Tool Use in ReAct

In this section, we will explore how to use our index to answer questions, and how to integrate retrieval as a tool within a `ReAct` agent.

## QA Strategy: Retrieve and Summarize

A simple yet effective strategy for Question Answering (QA) with RAG is to retrieve relevant documents and then pass them along with the question to an LLM to generate an answer. This is often called "Retrieve and Summarize" or "Stuffing".

In OneTwo, we have defined a specific interface for integrating QA strategies with indexes: `RetrievalQAStrategy`. We will create a strategy that inherits from `SimpleRetrievalQAStrategy` (a specialization of `RetrievalQAStrategy` for string queries and answers). This ensures consistency and allows our strategy to be used wherever a `RetrievalQAStrategy` is expected.

In [ ]:
# Define a simple retrieve and summarize strategy inheriting from SimpleRetrievalQAStrategy.
class RetrieveAndSummarize(retrieval_qa.SimpleRetrievalQAStrategy):
  """A simple retrieve and summarize strategy."""

  @ot.make_executable(copy_self=False, non_copied_args=["retriever"])
  async def __call__(
      self,
      question: str,
      retriever: retrieval.Retriever[str, retrieval_data_structures.Document],
  ) -> str:
    # 1. Retrieve relevant documents.
    docs = await retriever.retrieve(question)

    # 2. Format the context.
    context = "\n\n".join(
        [f"Document {i}:\n{d.content}" for i, d in enumerate(docs)]
    )

    # 3. Construct the prompt.
    prompt = f"""Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context:
{context}

Question: {question}
Answer:"""

    # 4. Call the LLM.
    # We use llm.instruct for simple instruction following.
    return await llm.instruct(prompt)


# Instantiate the strategy.
qa_strategy = RetrieveAndSummarize()

# Test it with our advanced index.
query = "What is the largest land animal and where can I find it in the zoo?"
answer = ot.run(qa_strategy(query, advanced_index))

print(f"Query: {query}\n")
print(f"Answer:\n{answer}")

# Retrieval as a Tool in ReAct

For more complex tasks, where the information needed is not available in a single retrieval step, or when we need to combine retrieval with other tools (like a calculator or execution sandbox), we can use a `ReAct` agent.

In OneTwo, we can easily wrap a retriever as a `Tool` and provide it to the `ReActAgent`.

In [ ]:
# Define the retrieval function.
def retrieve_zoo_info(query: str) -> str:
  """Retrieves facts about animals in the zoo."""
  # We use the advanced_index we created earlier.
  docs = ot.run(advanced_index.retrieve(query, max_results=4))
  return "\n".join([f"- {d.content}" for d in docs])


# Wrap it as a Tool.
zoo_retrieval_tool = llm_tool_use.Tool(
    name="retrieve_zoo_info",
    function=retrieve_zoo_info,
    description="Retrieves facts about animals in the zoo based on a query.",
    example="retrieve_zoo_info('elephant')",
)

# Define a custom exemplar for the ReAct agent to help it understand the format
# and how to use our specific tool.
custom_exemplar = react.ReActState(
    inputs="What does the lion eat?",
    updates=[
        react.ReActStep(
            thought=(
                "I need to find out what the lion eats. I can use the"
                " retrieve_zoo_info tool."
            ),
            action=llm_tool_use.FunctionCall(
                function_name="retrieve_zoo_info",
                args=("lion diet",),
                kwargs={},
            ),
            observation="Lions are carnivores and eat meat.",
            fmt=llm_tool_use.ArgumentFormat.PYTHON,
        ),
        react.ReActStep(
            is_finished=True,
            thought="I found that lions eat meat.",
            action=None,
            observation="Meat",
            fmt=llm_tool_use.ArgumentFormat.PYTHON,
        ),
    ],
)

# Configure the ReAct Agent.
config = python_tool_use.PythonToolUseEnvironmentConfig(
    tools=[zoo_retrieval_tool]
)

# We pass our custom exemplar to guide the agent.
agent = react.ReActAgent(
    environment_config=config,
    max_steps=5,
    exemplars=[custom_exemplar],
)

# Run the agent and request the final state.
query = "Are monkeys smart?"
output, final_state = ot.run(agent(inputs=query, return_final_state=True))

print(f"Query: {query}\n")
print(f"Final Output: {output}\n")

print("Execution Trace:")
for i, step in enumerate(final_state.updates):
  print(f"\nStep {i+1}:")
  if step.thought:
    print(f"Thought: {step.thought}")
  if step.is_finished:
    print(f"Final Answer: {step.observation}")
    break
  if step.action:
    print(f"Action: {step.action.function_name}({step.action.args})")
  if step.observation:
    print(f"Observation:\n {step.observation}")


# Loading HotpotQA

[HotpotQA](https://hotpotqa.github.io/) is a multi-hop question answering dataset where each question requires reasoning over multiple Wikipedia paragraphs to arrive at the answer.

OneTwo provides `HotpotQADatasetLoader`, a convenient wrapper that loads HotpotQA via TFDS and returns:

- **Examples**: Dictionaries with `question` and `answer` keys, compatible with `ot.evaluate`.
- **Documents**: `Document` objects (with `doc_id`, `title`, and `content` fields) representing the Wikipedia paragraphs associated with each example, suitable for building a retrieval corpus.

Note that Documents should not be expected to be in a particular order matching the order of the examples.

## Create the Loader

We create a `HotpotQADatasetLoader` with `max_number_of_examples=5` to load only the first 5 examples (and their associated documents) for a quick preview.

In [ ]:
loader = datasets.HotpotQADatasetLoader(
    split='validation',
    max_number_of_examples=5,
)

## Load Examples

The `load_examples()` method returns evaluation-compatible example dictionaries. Each example contains:
- `question`: The question string.
- `answer`: The gold answer string.
- `metadata`: Additional HotpotQA-specific fields (e.g., `id`, `type`, `level`).

In [ ]:
examples = ot.run(loader.load_examples())
print(f'Loaded {len(examples)} examples.\n')

for i, example in enumerate(examples):
  print(f'--- Example {i + 1} ---')
  print(f'  Question: {example["question"]}')
  print(f'  Answer:   {example["answer"]}')
  print(f'  Metadata: {example["metadata"]}')
  print()

## Load Documents

The `load_documents()` method returns `Document` objects extracted from the HotpotQA context paragraphs. Documents are **deduplicated** by `doc_id` (since the same Wikipedia paragraph can appear across multiple examples).

Each `Document` has:
- `doc_id`: Unique identifier (the Wikipedia paragraph title).
- `title`: The paragraph title.
- `content`: The concatenated text of the paragraph sentences.

In [ ]:
documents = ot.run(loader.load_documents())
print(f'Loaded {len(documents)} unique documents.\n')

for i, doc in enumerate(documents[:5]):
  print(f'--- Document {i + 1} ---')
  print(f'  Title:    {doc.title}')
  print(
      '  Content: '
      f' {doc.content[:200]}{"..." if len(doc.content) > 200 else ""}'
  )
  print(f'  Metadata: {doc.metadata}')

In [ ]:
cached_backends.save_caches()